# TP — Modèles d'Attention et Transformers pour le NLP

## Objectifs
Ce TP couvre deux tâches principales :
1. **Traduction automatique** : Transformer Seq2Seq Anglais → Français
2. **Génération de texte** : GPT-style pour descriptions de produits Amazon

---
**Contenu du notebook :**
- Partie 0 : Installation & Imports
- Partie 1 : Traduction automatique (Transformer Seq2Seq)
- Partie 2 : Génération de texte (Transformer Decoder-only)
- Partie 3 : Comparaison & Analyse des résultats

---
# Partie 0 — Installation et Imports

In [ ]:
# Installation des dépendances
!pip install torch torchtext datasets transformers sentencepiece sacrebleu nltk -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import math
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from collections import Counter

from datasets import load_dataset
import nltk
nltk.download('punkt', quiet=True)

# Seed pour la reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device utilisé : {device}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')

---
# Partie 1 — Traduction Automatique (Anglais → Français)

## Architecture : Transformer Seq2Seq (Vaswani et al., 2017)

Nous allons implémenter un Transformer complet avec :
- **Encoder** : multi-head self-attention + feed-forward
- **Decoder** : masked self-attention + cross-attention + feed-forward
- **Positional Encoding** : encodage sinusoïdal des positions

## 1.1 — Chargement et Préparation des Données (WMT14 subset)

In [ ]:
# Chargement d'un sous-ensemble de WMT14 (fr-en)
# Pour ce TP, on utilise un échantillon pour accélérer l'entraînement
print('Chargement du dataset WMT14 (sous-ensemble fr-en)...')

dataset = load_dataset('wmt14', 'fr-en', split='train', streaming=True)

# Collecte d'un échantillon de 50 000 paires (ajustable)
N_SAMPLES = 50_000
en_sentences, fr_sentences = [], []

for i, sample in enumerate(dataset):
    if i >= N_SAMPLES:
        break
    en_sentences.append(sample['translation']['en'])
    fr_sentences.append(sample['translation']['fr'])

print(f'Nombre de paires chargées : {len(en_sentences)}')
print(f'\nExemples :')
for i in range(3):
    print(f'  EN: {en_sentences[i]}')
    print(f'  FR: {fr_sentences[i]}')
    print()

In [ ]:
# ─────────────────────────────────────────────
# Tokenisation simple (word-level)
# ─────────────────────────────────────────────

def tokenize(text):
    """Tokenisation basique minuscule + split."""
    return text.lower().split()

class Vocabulary:
    """Construit un vocabulaire à partir d'une liste de phrases."""
    
    PAD_TOKEN = '<pad>'
    UNK_TOKEN = '<unk>'
    SOS_TOKEN = '<sos>'
    EOS_TOKEN = '<eos>'
    SPECIALS  = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

    def __init__(self, sentences, max_vocab=10_000, min_freq=2):
        counter = Counter(tok for sent in sentences for tok in tokenize(sent))
        vocab   = [w for w, c in counter.most_common() if c >= min_freq]
        vocab   = vocab[:max_vocab - len(self.SPECIALS)]

        self.itos = self.SPECIALS + vocab
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def __len__(self):  return len(self.itos)

    def encode(self, text, add_special=True):
        toks = tokenize(text)
        ids  = [self.stoi.get(t, self.stoi[self.UNK_TOKEN]) for t in toks]
        if add_special:
            ids = [self.stoi[self.SOS_TOKEN]] + ids + [self.stoi[self.EOS_TOKEN]]
        return ids

    def decode(self, ids, skip_special=True):
        toks = [self.itos[i] for i in ids]
        if skip_special:
            toks = [t for t in toks if t not in self.SPECIALS]
        return ' '.join(toks)


print('Construction des vocabulaires...')
src_vocab = Vocabulary(en_sentences, max_vocab=10_000, min_freq=2)
tgt_vocab = Vocabulary(fr_sentences, max_vocab=10_000, min_freq=2)
print(f'Taille vocabulaire EN : {len(src_vocab)}')
print(f'Taille vocabulaire FR : {len(tgt_vocab)}')

In [ ]:
# ─────────────────────────────────────────────
# Dataset PyTorch + collate_fn (padding dynamique)
# ─────────────────────────────────────────────

MAX_LEN = 50   # longueur maximale de séquence

class TranslationDataset(Dataset):
    def __init__(self, src_sents, tgt_sents, src_vocab, tgt_vocab, max_len=MAX_LEN):
        self.pairs = []
        for s, t in zip(src_sents, tgt_sents):
            src_ids = src_vocab.encode(s)
            tgt_ids = tgt_vocab.encode(t)
            if len(src_ids) <= max_len and len(tgt_ids) <= max_len:
                self.pairs.append((src_ids, tgt_ids))

    def __len__(self):  return len(self.pairs)
    def __getitem__(self, idx): return self.pairs[idx]


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)
    pad = 0  # index du token <pad>
    src_max = max(len(s) for s in src_batch)
    tgt_max = max(len(t) for t in tgt_batch)
    src_padded = torch.tensor([s + [pad]*(src_max-len(s)) for s in src_batch], dtype=torch.long)
    tgt_padded = torch.tensor([t + [pad]*(tgt_max-len(t)) for t in tgt_batch], dtype=torch.long)
    return src_padded, tgt_padded


# Split train / validation
split = int(0.9 * len(en_sentences))
train_ds = TranslationDataset(en_sentences[:split], fr_sentences[:split], src_vocab, tgt_vocab)
val_ds   = TranslationDataset(en_sentences[split:], fr_sentences[split:], src_vocab, tgt_vocab)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f'Paires train : {len(train_ds)} | val : {len(val_ds)}')

## 1.2 — Architecture Transformer

In [ ]:
# ─────────────────────────────────────────────
# 1. Positional Encoding (sinusoïdal)
# ─────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    """
    PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
    PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
    """
    def __init__(self, d_model, dropout=0.1, max_len=5_000):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe  = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)           # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x : (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


# ─────────────────────────────────────────────
# 2. Multi-Head Attention
# ─────────────────────────────────────────────

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k    = d_model // n_heads
        self.n_heads = n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def scaled_dot_product(self, Q, K, V, mask=None):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        return torch.matmul(attn, V), attn

    def split_heads(self, x):
        B, L, D = x.size()
        return x.view(B, L, self.n_heads, self.d_k).transpose(1, 2)

    def forward(self, Q, K, V, mask=None):
        B = Q.size(0)
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        x, self.attn_weights = self.scaled_dot_product(Q, K, V, mask)
        x = x.transpose(1, 2).contiguous().view(B, -1, self.n_heads * self.d_k)
        return self.W_o(x)


# ─────────────────────────────────────────────
# 3. Feed-Forward Network
# ─────────────────────────────────────────────

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
    def forward(self, x): return self.net(x)


# ─────────────────────────────────────────────
# 4. Encoder Layer
# ─────────────────────────────────────────────

class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff         = FeedForward(d_model, d_ff, dropout)
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, src_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, src_mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


# ─────────────────────────────────────────────
# 5. Decoder Layer
# ─────────────────────────────────────────────

class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff         = FeedForward(d_model, d_ff, dropout)
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.norm3      = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)

    def forward(self, x, enc_out, src_mask, tgt_mask):
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, tgt_mask)))
        x = self.norm2(x + self.dropout(self.cross_attn(x, enc_out, enc_out, src_mask)))
        x = self.norm3(x + self.dropout(self.ff(x)))
        return x


# ─────────────────────────────────────────────
# 6. Transformer complet
# ─────────────────────────────────────────────

class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size,
                 d_model=256, n_heads=8, n_layers=3,
                 d_ff=512, dropout=0.1, max_len=5_000):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab_size, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)
        self.scale   = math.sqrt(d_model)

        self.encoder = nn.ModuleList([EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.fc_out  = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def make_src_mask(self, src, pad_idx=0):
        # (B, 1, 1, src_len)
        return (src != pad_idx).unsqueeze(1).unsqueeze(2)

    def make_tgt_mask(self, tgt, pad_idx=0):
        B, T = tgt.size()
        pad_mask   = (tgt != pad_idx).unsqueeze(1).unsqueeze(2)      # (B,1,1,T)
        causal_mask= torch.tril(torch.ones(T, T, device=tgt.device)).bool()  # (T,T)
        return pad_mask & causal_mask

    def encode(self, src):
        src_mask = self.make_src_mask(src).to(src.device)
        x = self.pos_enc(self.src_emb(src) * self.scale)
        for layer in self.encoder:
            x = layer(x, src_mask)
        return x, src_mask

    def decode(self, tgt, enc_out, src_mask):
        tgt_mask = self.make_tgt_mask(tgt).to(tgt.device)
        x = self.pos_enc(self.tgt_emb(tgt) * self.scale)
        for layer in self.decoder:
            x = layer(x, enc_out, src_mask, tgt_mask)
        return x

    def forward(self, src, tgt):
        enc_out, src_mask = self.encode(src)
        dec_out           = self.decode(tgt, enc_out, src_mask)
        return self.fc_out(dec_out)


# Instanciation du modèle
model = Transformer(
    src_vocab_size=len(src_vocab),
    tgt_vocab_size=len(tgt_vocab),
    d_model=256, n_heads=8, n_layers=3,
    d_ff=512, dropout=0.1
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Nombre de paramètres : {n_params:,}')
print(model)

## 1.3 — Entraînement

In [ ]:
# ─────────────────────────────────────────────
# Optimiseur avec planification du learning rate
# (Noam scheduler — "Attention is All You Need")
# ─────────────────────────────────────────────

class NoamScheduler:
    def __init__(self, optimizer, d_model, warmup_steps=4000):
        self.optimizer    = optimizer
        self.d_model      = d_model
        self.warmup_steps = warmup_steps
        self._step        = 0
        self._lr          = 0

    def step(self):
        self._step += 1
        self._lr = (self.d_model ** -0.5) * min(
            self._step ** -0.5,
            self._step * self.warmup_steps ** -1.5
        )
        for pg in self.optimizer.param_groups:
            pg['lr'] = self._lr
        self.optimizer.step()


optimizer = optim.Adam(model.parameters(), betas=(0.9, 0.98), eps=1e-9)
scheduler = NoamScheduler(optimizer, d_model=256)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad>


def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for src, tgt in loader:
        src, tgt = src.to(device), tgt.to(device)
        tgt_in  = tgt[:, :-1]   # input  decoder (sans dernier token)
        tgt_out = tgt[:, 1:]    # target decoder (sans <sos>)

        optimizer.zero_grad()
        logits = model(src, tgt_in)           # (B, T-1, vocab)
        loss   = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(device), tgt.to(device)
            tgt_in  = tgt[:, :-1]
            tgt_out = tgt[:, 1:]
            logits  = model(src, tgt_in)
            loss    = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
            total_loss += loss.item()
    return total_loss / len(loader)


# ─────────────────────────────────────────────
# Boucle d'entraînement
# ─────────────────────────────────────────────

N_EPOCHS   = 10
best_val   = float('inf')
train_losses, val_losses = [], []

print(f'Entraînement sur {N_EPOCHS} epochs...')
for epoch in range(1, N_EPOCHS + 1):
    t0       = time.time()
    tr_loss  = train_epoch(model, train_loader)
    val_loss = evaluate(model, val_loader)
    elapsed  = time.time() - t0

    train_losses.append(tr_loss)
    val_losses.append(val_loss)

    if val_loss < best_val:
        best_val = val_loss
        torch.save(model.state_dict(), 'best_translation_model.pt')

    print(f'Epoch {epoch:02d}/{N_EPOCHS} | '
          f'Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | '
          f'Perplexité train: {math.exp(tr_loss):.2f} | '
          f'Temps: {elapsed:.1f}s')

In [ ]:
# Courbes de perte
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(1, len(train_losses)+1)
axes[0].plot(epochs, train_losses, 'b-o', label='Train')
axes[0].plot(epochs, val_losses,   'r-o', label='Validation')
axes[0].set(title='Loss (Cross-Entropy)', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, [math.exp(l) for l in train_losses], 'b-o', label='Train')
axes[1].plot(epochs, [math.exp(l) for l in val_losses],   'r-o', label='Validation')
axes[1].set(title='Perplexité', xlabel='Epoch', ylabel='Perplexité')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('translation_training_curves.png', dpi=150)
plt.show()
print('Courbes sauvegardées.')

## 1.4 — Inférence et Évaluation (BLEU)

In [ ]:
# ─────────────────────────────────────────────
# Décodage greedy
# ─────────────────────────────────────────────

def translate(model, sentence, src_vocab, tgt_vocab, max_len=50):
    model.eval()
    src = torch.tensor(src_vocab.encode(sentence)).unsqueeze(0).to(device)
    enc_out, src_mask = model.encode(src)

    tgt = torch.tensor([[tgt_vocab.stoi[tgt_vocab.SOS_TOKEN]]]).to(device)
    eos_idx = tgt_vocab.stoi[tgt_vocab.EOS_TOKEN]

    for _ in range(max_len):
        dec_out = model.decode(tgt, enc_out, src_mask)
        next_id = model.fc_out(dec_out[:, -1]).argmax(-1).item()
        if next_id == eos_idx:
            break
        tgt = torch.cat([tgt, torch.tensor([[next_id]]).to(device)], dim=1)

    return tgt_vocab.decode(tgt[0].tolist())


# Exemples de traduction
model.load_state_dict(torch.load('best_translation_model.pt', map_location=device))

test_sentences = [
    "Hello, how are you?",
    "The cat is on the table.",
    "I love machine learning and natural language processing.",
    "This is a great day for science.",
    "The transformer model has revolutionized NLP."
]

print('=== Exemples de traduction EN → FR ===\n')
for sent in test_sentences:
    translation = translate(model, sent, src_vocab, tgt_vocab)
    print(f'  EN : {sent}')
    print(f'  FR : {translation}')
    print()

In [ ]:
# ─────────────────────────────────────────────
# Score BLEU sur l'ensemble de validation
# ─────────────────────────────────────────────

from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

references, hypotheses = [], []
n_eval = min(500, len(val_ds))

print(f'Calcul du score BLEU sur {n_eval} paires...')
for src_ids, tgt_ids in val_ds.pairs[:n_eval]:
    src_sent = src_vocab.decode(src_ids)
    tgt_ref  = tgt_vocab.decode(tgt_ids).split()
    pred     = translate(model, src_sent, src_vocab, tgt_vocab).split()
    references.append([tgt_ref])
    hypotheses.append(pred)

sf   = SmoothingFunction().method4
bleu = corpus_bleu(references, hypotheses, smoothing_function=sf)
print(f'\nScore BLEU (validation) : {bleu*100:.2f}')

In [ ]:
# ─────────────────────────────────────────────
# Visualisation des poids d'attention
# ─────────────────────────────────────────────

def plot_attention(sentence, translation, attention, head=0):
    src_toks = ['<sos>'] + tokenize(sentence) + ['<eos>']
    tgt_toks = ['<sos>'] + tokenize(translation) + ['<eos>']

    # attention : (n_heads, tgt_len, src_len)
    attn = attention[head].cpu().detach().numpy()
    attn = attn[:len(tgt_toks), :len(src_toks)]

    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.matshow(attn, cmap='viridis')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(src_toks)))
    ax.set_yticks(range(len(tgt_toks)))
    ax.set_xticklabels(src_toks, rotation=90, fontsize=9)
    ax.set_yticklabels(tgt_toks, fontsize=9)
    ax.set(title=f'Cross-Attention — Head {head}',
           xlabel='Source (EN)', ylabel='Cible (FR)')
    plt.tight_layout()
    plt.savefig(f'attention_head{head}.png', dpi=150)
    plt.show()


# Extraction des poids du dernier décodeur
src_sent = "The cat sits on the mat."
model.eval()
src = torch.tensor(src_vocab.encode(src_sent)).unsqueeze(0).to(device)
enc_out, src_mask = model.encode(src)

fr_sent = translate(model, src_sent, src_vocab, tgt_vocab)
tgt = torch.tensor([tgt_vocab.stoi[tgt_vocab.SOS_TOKEN]]).unsqueeze(0).to(device)
eos_idx = tgt_vocab.stoi[tgt_vocab.EOS_TOKEN]
for _ in range(len(fr_sent.split()) + 2):
    dec_out = model.decode(tgt, enc_out, src_mask)
    next_id = model.fc_out(dec_out[:, -1]).argmax(-1).item()
    if next_id == eos_idx: break
    tgt = torch.cat([tgt, torch.tensor([[next_id]]).to(device)], dim=1)

# Récupération des poids cross-attention du dernier layer
cross_attn_weights = model.decoder[-1].cross_attn.attn_weights[0]  # (n_heads, T, S)

print(f'EN : {src_sent}')
print(f'FR : {fr_sent}')
for head in range(min(4, model.decoder[-1].cross_attn.n_heads)):
    plot_attention(src_sent, fr_sent, cross_attn_weights, head=head)

---
# Partie 2 — Génération de Texte (Descriptions de Produits Amazon)

## Architecture : Transformer Decoder-Only (style GPT)

Nous allons entraîner un modèle autorégressif pour générer des descriptions de produits à partir d'un titre.

## 2.1 — Chargement des Données Amazon Product

In [ ]:
# Amazon Product Reviews Dataset (McAuley Labs)
# On utilise le sous-ensemble "Electronics" disponible via HuggingFace

print('Chargement du dataset Amazon Product Data...')

try:
    # Dataset Amazon disponible sur HuggingFace
    amazon_ds = load_dataset(
        'McAuley-Lab/Amazon-Reviews-2023',
        'raw_meta_Electronics',
        split='full',
        streaming=True,
        trust_remote_code=True
    )
    source = 'amazon_hf'
except Exception:
    print('Dataset Amazon non accessible, génération de données synthétiques...')
    source = 'synthetic'


if source == 'amazon_hf':
    N_AMAZON = 20_000
    titles, descriptions = [], []
    for i, item in enumerate(amazon_ds):
        if i >= N_AMAZON: break
        title = item.get('title', '').strip()
        desc  = item.get('description', [])
        if isinstance(desc, list): desc = ' '.join(desc)
        desc = desc.strip()
        if title and desc and 20 < len(desc.split()) < 150:
            titles.append(title)
            descriptions.append(desc)
    print(f'Paires (titre, description) chargées : {len(titles)}')

else:
    # Données synthétiques représentatives
    SAMPLE_PRODUCTS = [
        ("Wireless Bluetooth Headphones",
         "Experience premium sound quality with our advanced wireless headphones. "
         "Features active noise cancellation, 30-hour battery life, and comfortable over-ear design. "
         "Perfect for music lovers and remote workers alike."),
        ("Stainless Steel Water Bottle",
         "Stay hydrated with our durable double-wall insulated water bottle. "
         "Keeps drinks cold for 24 hours and hot for 12 hours. BPA-free and eco-friendly."),
        ("Mechanical Gaming Keyboard",
         "Elevate your gaming setup with this RGB mechanical keyboard featuring tactile switches. "
         "Anti-ghosting technology ensures every keystroke is registered. Compact tenkeyless design."),
        ("4K Action Camera",
         "Capture your adventures in stunning 4K resolution. Waterproof up to 30 meters, "
         "comes with wide-angle lens and built-in image stabilization. Perfect for extreme sports."),
        ("Smart Fitness Tracker",
         "Monitor your health 24/7 with this advanced fitness band. Tracks heart rate, sleep quality, "
         "steps, and calories. Compatible with iOS and Android. 7-day battery life."),
    ]
    # Augmentation par variations aléatoires
    adjectives = ['premium', 'advanced', 'durable', 'portable', 'innovative', 'professional', 'smart']
    titles, descriptions = [], []
    for title, desc in SAMPLE_PRODUCTS * 4000:
        adj = random.choice(adjectives)
        titles.append(f'{adj.capitalize()} {title}')
        descriptions.append(desc)
    random.shuffle(list(zip(titles, descriptions)))
    print(f'Données synthétiques générées : {len(titles)} paires')

print(f'\nExemple :')
print(f'  Titre       : {titles[0]}')
print(f'  Description : {descriptions[0][:100]}...')

In [ ]:
# ─────────────────────────────────────────────
# Vocabulaire unique (titre + description)
# Format d'entrée : <sos> TITLE <sep> DESCRIPTION <eos>
# ─────────────────────────────────────────────

class ProductVocabulary(Vocabulary):
    """Vocabulary enrichi avec un token <sep>."""
    SEP_TOKEN = '<sep>'
    SPECIALS  = Vocabulary.SPECIALS + ['<sep>']

    def __init__(self, all_texts, max_vocab=15_000, min_freq=2):
        counter = Counter(tok for t in all_texts for tok in tokenize(t))
        vocab   = [w for w, c in counter.most_common() if c >= min_freq]
        vocab   = vocab[:max_vocab - len(self.SPECIALS)]
        self.itos = self.SPECIALS + vocab
        self.stoi = {w: i for i, w in enumerate(self.itos)}

    def encode_pair(self, title, desc):
        """Encode : <sos> title <sep> description <eos>"""
        t_ids = [self.stoi.get(w, self.stoi[self.UNK_TOKEN]) for w in tokenize(title)]
        d_ids = [self.stoi.get(w, self.stoi[self.UNK_TOKEN]) for w in tokenize(desc)]
        return ([self.stoi[self.SOS_TOKEN]]
                + t_ids
                + [self.stoi[self.SEP_TOKEN]]
                + d_ids
                + [self.stoi[self.EOS_TOKEN]])


all_texts  = titles + descriptions
prod_vocab = ProductVocabulary(all_texts, max_vocab=15_000)
SEP_IDX    = prod_vocab.stoi[prod_vocab.SEP_TOKEN]

print(f'Taille vocabulaire produits : {len(prod_vocab)}')


# Dataset
MAX_PROD_LEN = 128

class ProductDataset(Dataset):
    def __init__(self, titles, descs, vocab, max_len=MAX_PROD_LEN):
        self.seqs = []
        for t, d in zip(titles, descs):
            ids = vocab.encode_pair(t, d)
            if len(ids) <= max_len:
                self.seqs.append(ids)

    def __len__(self):  return len(self.seqs)
    def __getitem__(self, i): return self.seqs[i]


def prod_collate(batch):
    max_l = max(len(s) for s in batch)
    padded = [s + [0]*(max_l - len(s)) for s in batch]
    return torch.tensor(padded, dtype=torch.long)


split2     = int(0.9 * len(titles))
train_pd   = ProductDataset(titles[:split2], descriptions[:split2], prod_vocab)
val_pd     = ProductDataset(titles[split2:], descriptions[split2:], prod_vocab)
train_pl   = DataLoader(train_pd, batch_size=64, shuffle=True,  collate_fn=prod_collate)
val_pl     = DataLoader(val_pd,   batch_size=64, shuffle=False, collate_fn=prod_collate)

print(f'Séquences train : {len(train_pd)} | val : {len(val_pd)}')

## 2.2 — Architecture GPT (Decoder-Only Transformer)

In [ ]:
# ─────────────────────────────────────────────
# Transformer Decoder-Only (style GPT)
# Utilise uniquement des couches d'auto-attention
# avec masque causal ("no peeking")
# ─────────────────────────────────────────────

class GPTBlock(nn.Module):
    """Un bloc Transformer decoder-only."""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn    = MultiHeadAttention(d_model, n_heads, dropout)
        self.ff      = FeedForward(d_model, d_ff, dropout)
        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, causal_mask):
        x = self.norm1(x + self.dropout(self.attn(x, x, x, causal_mask)))
        x = self.norm2(x + self.dropout(self.ff(x)))
        return x


class MiniGPT(nn.Module):
    """Transformer Decoder-Only pour la génération de texte."""
    def __init__(self, vocab_size, d_model=256, n_heads=8,
                 n_layers=4, d_ff=512, dropout=0.1, max_len=512):
        super().__init__()
        self.emb     = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)
        self.scale   = math.sqrt(d_model)
        self.blocks  = nn.ModuleList([
            GPTBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1: nn.init.xavier_uniform_(p)

    def make_causal_mask(self, x):
        B, T = x.size()
        pad_mask    = (x != 0).unsqueeze(1).unsqueeze(2)             # (B,1,1,T)
        causal_mask = torch.tril(torch.ones(T, T, device=x.device)).bool()
        return pad_mask & causal_mask

    def forward(self, x):
        mask = self.make_causal_mask(x)
        h    = self.pos_enc(self.emb(x) * self.scale)
        for block in self.blocks:
            h = block(h, mask)
        h = self.norm(h)
        return self.head(h)


gpt_model = MiniGPT(
    vocab_size=len(prod_vocab),
    d_model=256, n_heads=8, n_layers=4, d_ff=512, dropout=0.1
).to(device)

n_params_gpt = sum(p.numel() for p in gpt_model.parameters() if p.requires_grad)
print(f'Paramètres MiniGPT : {n_params_gpt:,}')

## 2.3 — Entraînement du modèle de génération

In [ ]:
gpt_optimizer = optim.AdamW(gpt_model.parameters(), lr=3e-4, weight_decay=0.01)
gpt_scheduler = optim.lr_scheduler.CosineAnnealingLR(gpt_optimizer, T_max=10)
gpt_criterion = nn.CrossEntropyLoss(ignore_index=0)


def gpt_train_epoch(model, loader):
    model.train()
    total = 0
    for batch in loader:
        batch = batch.to(device)
        inp, target = batch[:, :-1], batch[:, 1:]
        gpt_optimizer.zero_grad()
        logits = model(inp)
        loss   = gpt_criterion(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        gpt_optimizer.step()
        total += loss.item()
    gpt_scheduler.step()
    return total / len(loader)


def gpt_evaluate(model, loader):
    model.eval()
    total = 0
    with torch.no_grad():
        for batch in loader:
            batch  = batch.to(device)
            inp, target = batch[:, :-1], batch[:, 1:]
            logits = model(inp)
            loss   = gpt_criterion(logits.reshape(-1, logits.size(-1)), target.reshape(-1))
            total += loss.item()
    return total / len(loader)


GPT_EPOCHS = 10
gpt_train_losses, gpt_val_losses = [], []
best_gpt_val = float('inf')

print(f'Entraînement MiniGPT sur {GPT_EPOCHS} epochs...')
for epoch in range(1, GPT_EPOCHS + 1):
    t0 = time.time()
    tr = gpt_train_epoch(gpt_model, train_pl)
    vl = gpt_evaluate(gpt_model, val_pl)
    elapsed = time.time() - t0

    gpt_train_losses.append(tr)
    gpt_val_losses.append(vl)

    if vl < best_gpt_val:
        best_gpt_val = vl
        torch.save(gpt_model.state_dict(), 'best_gpt_model.pt')

    print(f'Epoch {epoch:02d}/{GPT_EPOCHS} | Train: {tr:.4f} | Val: {vl:.4f} | '
          f'PPL: {math.exp(vl):.2f} | {elapsed:.1f}s')

In [ ]:
# Courbes de perte GPT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
e = range(1, len(gpt_train_losses)+1)

axes[0].plot(e, gpt_train_losses, 'g-o', label='Train')
axes[0].plot(e, gpt_val_losses,   'm-o', label='Validation')
axes[0].set(title='MiniGPT — Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(e, [math.exp(l) for l in gpt_train_losses], 'g-o', label='Train')
axes[1].plot(e, [math.exp(l) for l in gpt_val_losses],   'm-o', label='Validation')
axes[1].set(title='MiniGPT — Perplexité', xlabel='Epoch', ylabel='PPL')
axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('gpt_training_curves.png', dpi=150)
plt.show()

## 2.4 — Génération de descriptions (Top-k, Top-p, Température)

In [ ]:
# ─────────────────────────────────────────────
# Stratégies de décodage :
#   - Greedy
#   - Temperature sampling
#   - Top-k sampling
#   - Top-p (nucleus) sampling
# ─────────────────────────────────────────────

def top_k_top_p_filter(logits, top_k=0, top_p=0.0):
    """Filtre les logits avec top-k et/ou top-p."""
    if top_k > 0:
        vals, _ = torch.topk(logits, top_k)
        logits[logits < vals[:, [-1]]] = float('-inf')
    if top_p > 0.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        probs = torch.softmax(sorted_logits, dim=-1)
        cumprobs = torch.cumsum(probs, dim=-1)
        remove = cumprobs - probs > top_p
        remove[:, 0] = False  # garder au moins 1 token
        sorted_logits[remove] = float('-inf')
        logits.scatter_(1, sorted_idx, sorted_logits)
    return logits


def generate_description(model, vocab, title, max_new_tokens=80,
                          temperature=1.0, top_k=50, top_p=0.9):
    model.eval()
    sep_idx = vocab.stoi[vocab.SEP_TOKEN]
    eos_idx = vocab.stoi[vocab.EOS_TOKEN]

    # Prompt = <sos> title <sep>
    title_ids = [vocab.stoi.get(w, vocab.stoi[vocab.UNK_TOKEN]) for w in tokenize(title)]
    ids = [vocab.stoi[vocab.SOS_TOKEN]] + title_ids + [sep_idx]
    x   = torch.tensor([ids], dtype=torch.long).to(device)

    generated = []
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits    = model(x)[:, -1, :] / temperature
            logits    = top_k_top_p_filter(logits, top_k=top_k, top_p=top_p)
            probs     = torch.softmax(logits, dim=-1)
            next_tok  = torch.multinomial(probs, 1).item()
            if next_tok == eos_idx: break
            generated.append(next_tok)
            x = torch.cat([x, torch.tensor([[next_tok]]).to(device)], dim=1)

    return vocab.decode(generated)


# Chargement du meilleur modèle
gpt_model.load_state_dict(torch.load('best_gpt_model.pt', map_location=device))

test_titles = [
    "Wireless Noise Cancelling Headphones",
    "Ultra-Slim Laptop 15 inch",
    "Professional Coffee Maker with Grinder",
    "Smart Home Security Camera",
    "Ergonomic Office Chair"
]

print('=== Génération de descriptions de produits ===\n')
for title in test_titles:
    desc = generate_description(gpt_model, prod_vocab, title,
                                  temperature=0.8, top_k=40, top_p=0.92)
    print(f'Produit    : {title}')
    print(f'Description: {desc}')
    print()

In [ ]:
# ─────────────────────────────────────────────
# Comparaison des stratégies de décodage
# ─────────────────────────────────────────────

title_demo = "Wireless Bluetooth Speaker"
print(f'Titre : "{title_demo}"\n')
print('─' * 60)

strategies = [
    {'name': 'Greedy (T=0.1)',        'temperature': 0.1, 'top_k': 1,  'top_p': 0.0},
    {'name': 'Temperature (T=0.7)',   'temperature': 0.7, 'top_k': 0,  'top_p': 0.0},
    {'name': 'Top-k (k=50, T=1.0)',   'temperature': 1.0, 'top_k': 50, 'top_p': 0.0},
    {'name': 'Top-p (p=0.9, T=1.0)',  'temperature': 1.0, 'top_k': 0,  'top_p': 0.9},
    {'name': 'Top-k+p (k=50, p=0.9)', 'temperature': 0.8, 'top_k': 50, 'top_p': 0.9},
]

for s in strategies:
    desc = generate_description(gpt_model, prod_vocab, title_demo,
                                  temperature=s['temperature'],
                                  top_k=s['top_k'], top_p=s['top_p'])
    print(f"[{s['name']}]")
    print(f"  {desc}")
    print()

---
# Partie 3 — Comparaison et Analyse des Résultats

In [ ]:
# ─────────────────────────────────────────────
# Tableau comparatif des deux modèles
# ─────────────────────────────────────────────

import pandas as pd

summary = {
    'Modèle': ['Transformer Seq2Seq', 'MiniGPT (Decoder-Only)'],
    'Tâche': ['Traduction EN→FR', 'Génération de descriptions'],
    'Architecture': ['Encoder + Decoder', 'Decoder uniquement'],
    'Paramètres': [f'{n_params:,}', f'{n_params_gpt:,}'],
    'Meilleure val. Loss': [f'{min(val_losses):.4f}', f'{min(gpt_val_losses):.4f}'],
    'Meilleure val. PPL':  [f'{math.exp(min(val_losses)):.2f}',
                            f'{math.exp(min(gpt_val_losses)):.2f}'],
    'Métrique': ['BLEU', 'Perplexité + Cohérence'],
}

df = pd.DataFrame(summary)
print(df.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────
# Visualisation comparative des perplexités
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
epochs = range(1, max(len(val_losses), len(gpt_val_losses)) + 1)

# Loss
ax = axes[0]
ax.plot(range(1, len(val_losses)+1),     val_losses,     'b-o', label='Seq2Seq (traduction)')
ax.plot(range(1, len(gpt_val_losses)+1), gpt_val_losses, 'm-s', label='MiniGPT (génération)')
ax.set(title='Comparaison — Validation Loss', xlabel='Epoch', ylabel='Loss')
ax.legend(); ax.grid(True, alpha=0.4)

# PPL
ax = axes[1]
ax.plot(range(1, len(val_losses)+1),
        [math.exp(l) for l in val_losses],     'b-o', label='Seq2Seq (traduction)')
ax.plot(range(1, len(gpt_val_losses)+1),
        [math.exp(l) for l in gpt_val_losses], 'm-s', label='MiniGPT (génération)')
ax.set(title='Comparaison — Perplexité', xlabel='Epoch', ylabel='PPL')
ax.legend(); ax.grid(True, alpha=0.4)

plt.suptitle('Analyse Comparative des Modèles Transformer', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('comparison_plot.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# Diagramme des composants du Transformer
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

def draw_box(ax, xy, w, h, text, color='#AED6F1', fontsize=9):
    from matplotlib.patches import FancyBboxPatch
    x, y = xy
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.05',
                          facecolor=color, edgecolor='#2C3E50', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x+w/2, y+h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', color='#2C3E50',
            wrap=True, multialignment='center')

# Modèle Seq2Seq
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 14); ax.axis('off')
ax.set_title('Transformer Seq2Seq\n(Traduction EN→FR)', fontsize=12, fontweight='bold', pad=10)

enc_blocks = [
    (1, 11.5, 3.5, 1.2, 'Input Embedding\n+ Positional Encoding', '#D5E8D4'),
    (1, 9.8, 3.5, 1.2, 'Multi-Head\nSelf-Attention', '#AED6F1'),
    (1, 8.1, 3.5, 1.2, 'Add & Norm', '#FAD7A0'),
    (1, 6.4, 3.5, 1.2, 'Feed-Forward\nNetwork', '#D2B4DE'),
    (1, 4.7, 3.5, 1.2, 'Add & Norm', '#FAD7A0'),
    (1, 3.0, 3.5, 1.0, 'ENCODER\n(N × couches)', '#FDEBD0'),
]
dec_blocks = [
    (5.5, 11.5, 3.5, 1.2, 'Output Embedding\n+ Positional Encoding', '#D5E8D4'),
    (5.5, 9.8, 3.5, 1.2, 'Masked Multi-Head\nSelf-Attention', '#AED6F1'),
    (5.5, 8.1, 3.5, 1.2, 'Add & Norm', '#FAD7A0'),
    (5.5, 6.4, 3.5, 1.2, 'Cross-Attention\n(Encoder-Decoder)', '#F1948A'),
    (5.5, 4.7, 3.5, 1.2, 'Feed-Forward\nNetwork', '#D2B4DE'),
    (5.5, 3.0, 3.5, 1.0, 'DECODER\n(N × couches)', '#FDEBD0'),
]
for x, y, w, h, t, c in enc_blocks + dec_blocks:
    draw_box(ax, (x, y), w, h, t, c)

draw_box(ax, (3.25, 1.2), 3.5, 1.2, 'Linear + Softmax\n→ Token FR', '#A9DFBF')

ax.annotate('', xy=(2.75, 3.0), xytext=(2.75, 2.0),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.annotate('', xy=(7.25, 3.0), xytext=(7.25, 2.0),
            arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
ax.annotate('EN', xy=(2.75, 1.0), ha='center', fontsize=10, color='#2C3E50')
ax.annotate('FR (prev)', xy=(7.25, 1.0), ha='center', fontsize=10, color='#2C3E50')

# Modèle GPT
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 14); ax.axis('off')
ax.set_title('MiniGPT (Decoder-Only)\n(Génération Amazon)', fontsize=12, fontweight='bold', pad=10)

gpt_blocks = [
    (3, 12, 4, 1.2, 'Input Embedding\n+ Positional Encoding', '#D5E8D4'),
    (3, 10.3, 4, 1.2, 'Masked Multi-Head\nSelf-Attention', '#AED6F1'),
    (3, 8.6, 4, 1.2, 'Add & Norm', '#FAD7A0'),
    (3, 6.9, 4, 1.2, 'Feed-Forward\nNetwork', '#D2B4DE'),
    (3, 5.2, 4, 1.2, 'Add & Norm', '#FAD7A0'),
    (3, 3.5, 4, 1.0, 'N × GPT Blocks', '#FDEBD0'),
    (3, 2.0, 4, 1.0, 'Layer Norm', '#FAD7A0'),
    (3, 0.5, 4, 1.2, 'Linear + Softmax\n→ Prochain Token', '#A9DFBF'),
]
for x, y, w, h, t, c in gpt_blocks:
    draw_box(ax, (x, y), w, h, t, c)

plt.tight_layout()
plt.savefig('transformer_architectures.png', dpi=150, bbox_inches='tight')
plt.show()
print('Diagrammes des architectures sauvegardés.')

---
# Conclusion

## Résumé des apprentissages

### Tâche 1 — Traduction Automatique (Seq2Seq Transformer)
- **Architecture** : Encoder-Decoder avec cross-attention, implémenté depuis zéro
- **Points clés** : positional encoding sinusoïdal, masques causaux, Noam scheduler
- **Évaluation** : score BLEU + visualisation des poids d'attention
- **Observation** : la cross-attention apprend des alignements source-cible interprétables

### Tâche 2 — Génération de Texte (MiniGPT)
- **Architecture** : Decoder-only avec auto-attention masquée (style GPT)
- **Points clés** : prompt conditioning via `<sep>`, décodage top-k/top-p/température
- **Observation** : la température contrôle créativité vs cohérence ; top-p améliore la qualité

### Comparaison Seq2Seq vs GPT
| | Seq2Seq | Decoder-Only |
|---|---|---|
| Tâche principale | Transduction conditionnelle | Génération autoregressive |
| Attention | Self + Cross | Self seulement |
| Entrée | Séquence source séparée | Tout dans une séquence |
| Flexibilité | Paires parallèles requises | Un seul flux de tokens |

## Pour aller plus loin
- Utiliser un tokenizer BPE (SentencePiece) pour réduire le vocabulaire
- Appliquer le fine-tuning sur des modèles pré-entraînés (mBART, GPT-2, BLOOM)
- Explorer les techniques d'alignement (beam search, diverse beam search)
- Tester l'évaluation par ROUGE (génération) et METEOR (traduction)